# Polymarket History — Quickstart with DuckDB

This notebook demonstrates how to query the Polymarket prediction market dataset
using DuckDB directly against local Parquet files.

**Attribution:** Data sourced from Polymarket via Pancake (usepancake.com)

**Point-in-time note:** Always gate on `released_at <= t`, not `observed_at <= t`,
to avoid lookahead bias. See the README for details.


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# Paths — adjust if running from a different directory
REPO_ROOT = Path('..').resolve()
DATASETS = REPO_ROOT / 'datasets' / 'v1' / 'polymarket'

con = duckdb.connect(database=':memory:')

print('DuckDB version:', duckdb.__version__)
print('Dataset root:  ', DATASETS)

# Create views for each table
for table in ['markets', 'outcomes', 'candles_1d', 'resolutions', 'trades']:
    parquet_glob = str(DATASETS / table / '**' / '*.parquet').replace('\\', '/')
    try:
        con.execute(f"CREATE OR REPLACE VIEW {table} AS SELECT * FROM read_parquet('{parquet_glob}', hive_partitioning=false)")
        count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'  {table:<14} {count:>10,} rows')
    except Exception as e:
        print(f'  {table}: NOT FOUND ({e})')

print('\nViews ready.')

DuckDB version: 1.5.3
Dataset root:   C:\Users\micha\pancake-production\polymarket-history\datasets\v1\polymarket
  markets            24,304 rows
  outcomes           15,796 rows
  candles_1d        851,131 rows
  resolutions         7,788 rows
  trades            132,868 rows

Views ready.


## 1. Biggest markets by total volume

The top-10 Polymarket markets ever by total traded volume in USD.

In [2]:
top_markets = con.execute("""
    SELECT
        market_id,
        title,
        category,
        status,
        ROUND(total_volume_usd / 1e6, 2) AS volume_m_usd
    FROM markets
    WHERE total_volume_usd IS NOT NULL
    ORDER BY total_volume_usd DESC
    LIMIT 10
""").df()

top_markets

,market_id,title,category,status,volume_m_usd
0,e83b4767-3664-4939-8746-a4d23f926a87,Will Donald Trump win the 2024 US Presidential...,politics,resolved,1531.48
1,0e7afa6e-b874-41f6-89b7-8c38c4ea3c83,Will Kamala Harris win the 2024 US Presidentia...,politics,resolved,1037.04
2,8fd29feb-7c2b-47e7-bb97-38e62255cb70,Will Donald Trump be inaugurated?,politics,resolved,400.41
3,fa6f1933-c6e5-4aa6-beee-21d7b14007df,Will the Sacramento Kings win the 2025 NBA Fin...,sports,resolved,378.01
4,8f277456-5739-4efc-a03a-57831863ca0f,Will Nicolae Ciucă win the 2024 Romanian Presi...,politics,resolved,326.51
5,8ee25e43-f6a1-49e6-bfee-104586a6aa8e,US forces enter Iran by April 30?,geopolitics,resolved,269.05
6,d2e5352e-15d1-4d01-8175-37fa971625cd,Will Zelenskyy wear a suit before July?,geopolitics,resolved,242.23
7,76662acf-8e93-4c5b-869d-f9d9f9af9294,Will any other Republican Politician win the 2...,politics,resolved,241.66
8,91936319-7e3b-41d2-95bb-a1257370c52e,Fed decreases interest rates by 50+ bps after ...,economics,resolved,235.07
9,d506887d-d4c0-4c4a-9da2-5a45f0027638,Fed increases interest rates by 25+ bps after ...,economics,resolved,216.46


## 2. Calibration curve sketch

A calibration curve answers: "when the market priced an event at X%, how often did it actually happen?"

We compute this from resolved binary markets:
- Take the closing price (last candle's `close`) for each resolved market's YES-side instrument.
- Bucket into 10pp-wide bins.
- Compare to actual resolution rate (fraction where YES won).


In [3]:
# Step 1: get resolved markets with their winning outcome instrument
# We need the LAST candle before resolution for each YES-side instrument
# 'winning_outcome_label' tells us if YES won

calibration_data = con.execute("""
    WITH resolved_binary AS (
        -- Get resolved markets that have resolutions with outcome labels
        SELECT
            m.market_id,
            r.winning_outcome_label,
            r.resolved_at
        FROM markets m
        JOIN resolutions r ON m.market_id = r.market_id
        WHERE m.market_kind = 'binary'
          AND r.winning_outcome_label IS NOT NULL
    ),
    last_candle AS (
        -- Last daily close for each YES-side instrument before resolution
        SELECT
            c.market_id,
            c.close,
            ROW_NUMBER() OVER (
                PARTITION BY c.market_id
                ORDER BY c.observed_at DESC
            ) AS rn
        FROM candles_1d c
        JOIN resolved_binary rb ON c.market_id = rb.market_id
        WHERE c.outcome_label ILIKE '%yes%'
          AND c.observed_at <= rb.resolved_at
    )
    SELECT
        rb.market_id,
        lc.close AS last_price,
        (rb.winning_outcome_label ILIKE '%yes%')::INTEGER AS yes_won,
        FLOOR(lc.close * 10) / 10 AS price_bucket
    FROM resolved_binary rb
    JOIN last_candle lc ON rb.market_id = lc.market_id AND lc.rn = 1
    WHERE lc.close BETWEEN 0.01 AND 0.99
""").df()

print(f'Markets with calibration data: {len(calibration_data):,}')
calibration_data.head()

Markets with calibration data: 1,440


,market_id,last_price,yes_won,price_bucket
0,03996865-b734-40e2-937f-c9633cb44398,0.4300,1,0.4
1,0840f654-5c66-4947-94db-b55fa09536eb,0.0615,0,0.0
2,0c4ee8c7-fc6d-4a75-bb13-df7dd5b74034,0.1210,0,0.1
3,0f0bb122-da2c-4ebd-807b-e6a077a221f3,0.7100,0,0.7
4,1399fc13-8084-402a-a92a-3fb05c5ad0d0,0.4130,1,0.4


In [4]:
# Aggregate into 10pp bins
import numpy as np

bins = np.arange(0, 1.05, 0.1)
labels = [f'{int(b*100)}-{int((b+0.1)*100)}%' for b in bins[:-1]]

calibration_data['bin'] = pd.cut(calibration_data['last_price'], bins=bins, labels=labels, include_lowest=True)

curve = calibration_data.groupby('bin', observed=True).agg(
    n_markets=('yes_won', 'count'),
    yes_rate=('yes_won', 'mean'),
    avg_price=('last_price', 'mean')
).reset_index()

print('Calibration curve (binary markets, last pre-resolution daily close vs actual YES rate):')
print(curve.to_string(index=False))

Calibration curve (binary markets, last pre-resolution daily close vs actual YES rate):
    bin  n_markets  yes_rate  avg_price
  0-10%        383  0.073107   0.037141
 10-20%        111  0.279279   0.149009
 20-30%        107  0.299065   0.248150
 30-40%        101  0.405941   0.354287
 40-50%        149  0.476510   0.456440
 50-60%        133  0.601504   0.547744
 60-70%        111  0.684685   0.652838
 70-80%        109  0.798165   0.750743
 80-90%         63  0.920635   0.849008
90-100%        173  0.965318   0.959321


## 3. Resolution outcomes by category

Which categories have the most resolved markets, and how many are YES vs NO?


In [5]:
outcomes_by_cat = con.execute("""
    SELECT
        COALESCE(m.category, '(uncategorized)') AS category,
        COUNT(*) AS resolved_markets,
        SUM(CASE WHEN r.winning_outcome_label ILIKE '%yes%' THEN 1 ELSE 0 END) AS yes_count,
        SUM(CASE WHEN r.winning_outcome_label ILIKE '%no%' THEN 1 ELSE 0 END) AS no_count,
        ROUND(
            100.0 * SUM(CASE WHEN r.winning_outcome_label ILIKE '%yes%' THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            1
        ) AS yes_pct
    FROM resolutions r
    JOIN markets m ON r.market_id = m.market_id
    WHERE r.winning_outcome_label IS NOT NULL
      AND m.market_kind = 'binary'
    GROUP BY 1
    ORDER BY resolved_markets DESC
    LIMIT 15
""").df()

outcomes_by_cat

,category,resolved_markets,yes_count,no_count,yes_pct
0,sports,3213,220.0,680.0,6.8
1,other,2136,409.0,1126.0,19.1
2,politics,1001,205.0,786.0,20.5
3,crypto,808,183.0,516.0,22.6
4,geopolitics,466,130.0,335.0,27.9
5,economics,156,39.0,117.0,25.0
6,(uncategorized),8,1.0,6.0,12.5


## 4. Price bar coverage over time

How many daily bars exist per month? Shows the growth of Polymarket from 2022 onward.


In [6]:
coverage = con.execute("""
    SELECT
        strftime(to_timestamp(observed_at), '%Y-%m') AS month,
        COUNT(*) AS bar_count,
        COUNT(DISTINCT market_id) AS unique_markets
    FROM candles_1d
    GROUP BY 1
    ORDER BY 1
""").df()

print(f'Months with data: {len(coverage)}')
print(f'Total bars: {coverage.bar_count.sum():,}')
print(f'Peak month: {coverage.loc[coverage.bar_count.idxmax(), "month"]} ({coverage.bar_count.max():,} bars)')
print()
print(coverage.to_string(index=False))

Months with data: 44
Total bars: 851,131
Peak month: 2025-12 (54,898 bars)

  month  bar_count  unique_markets
2022-11         18               1
2022-12         92               2
2023-01        160               3
2023-02        180               5
2023-03        324               6
2023-04        324               6
2023-05        316               6
2023-06        264               6
2023-07        320               6
2023-08        372               6
2023-09        400               8
2023-10        444               8
2023-11        486              10
2023-12        790              14
2024-01       4160             112
2024-02       6812             123
2024-03       9434             197
2024-04      12642             222
2024-05      14274             241
2024-06      15058             262
2024-07      19204             373
2024-08      21506             500
2024-09      28220             578
2024-10      39286             730
2024-11      35646             837
2024-12      3

---

## Want zero-setup backtesting on this data?

Visit **[usepancake.com](https://usepancake.com)** — Claude builds the strategy, Pancake runs and verifies it.
No data download required.

---
*Attribution: Data sourced from Polymarket via Pancake (usepancake.com) — CC BY 4.0*
